# 11 — チューニングと数式変更の実践手順

## 最終到達目標
現象を「どの層の、どの式の、どの残差か」へ戻し、一度に1仮説だけ変更する。
この章は魔法の推奨値ではなく、再現可能な変更手順を作る。

照合対象: [`legged_control` `a7f381c036`](https://github.com/qiayuanliao/legged_control/tree/a7f381c0367e98e31c01336e678eef47e304d40d)


## 検証範囲に関する必須注記

このprojectでは **ROS2 portを作成・compile・実行していない**。したがってROS2 parityは
**NOT VERIFIED / FAIL-CLOSED** である。上流commit `a7f381c0367e98e31c01336e678eef47e304d40d` はROS1実装であり、
project所有MuJoCo adapterはOCS2のhorizon SQPを瞬時force plannerへ、
Pinocchio/qpOASES WBCをMuJoCo acceleration inverse dynamicsへ置換し、
元のestimator/hardware経路も持たない。保存済み30 scenario dataが示すのはadapter挙動だけで、
上流 `legged_control` の性能でもROS2移行の検証でもない。


In [1]:
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`from pathlib import Path` の依存を明示して再現可能な実行環境を作る。
from pathlib import Path
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`import numpy as np` の依存を明示して再現可能な実行環境を作る。
import numpy as np
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`import matplotlib.pyplot as plt` の依存を明示して再現可能な実行環境を作る。
import matplotlib.pyplot as plt

# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = Path.cwd()` の演算・変換をPythonで評価する。
ROOT = Path.cwd()
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`for candidate in [ROOT, *ROOT.parents]:` の反復範囲を固定して各sampleを処理する。 数式: `for candidate in [ROOT, *ROOT.parents]:` の演算・変換をPythonで評価する。
for candidate in [ROOT, *ROOT.parents]:
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`if (candidate / "pyproject.toml").exists():` の条件で安全側の実行分岐を選ぶ。 数式: `if (candidate / "pyproject.toml").exists():` の演算・変換をPythonで評価する。
    if (candidate / "pyproject.toml").exists():
        # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = candidate` の演算・変換をPythonで評価する。
        ROOT = candidate
        # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`break` をこの章の処理順に沿って実行する。
        break

# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`np.set_printoptions(precision` を後続計算で使う明示的な中間量として設定する。 数式: `np.set_printoptions(precision=4, suppress=True)` の演算・変換をPythonで評価する。
np.set_printoptions(precision=4, suppress=True)
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、直前の式・構造へ `plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})` の要素または終端を対応付ける。
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`print("repository:", ROOT)` の観測値を表示して判定根拠を残す。
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


## 変更の順序
1. baseline commit/config、robot、gait、指令、床、seedを固定
2. 失敗を層へ分類: sensor/frame → estimator → reference/gait → NMPC → WBC → joint/HW
3. 数式の単位・符号・shapeを手計算と静的testで確認
4. 変更parameterは1群、式変更は1項だけ
5. constraint residual、solver status/time、tracking、torque、slipを保存
6. 改善と副作用を比較し、戻せる差分にする

### 症状から最初に見る場所
- 静止でbase位置drift: IMU重力/frame、接触flag、KF noise
- 横滑り: 実摩擦、NMPC円錐margin、WBC pyramid、接触誤判定
- 足先が遅れる: swing task residual、WBC weight、torque saturation
- 姿勢追従が弱い: reference、Q、feasibility、base task weight
- torque振動: policy age、接触切替、WBC active set、Kd、delay


In [2]:
# 変更記録を機械的に比較するための最小schema。
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`baseline` を後続計算で使う明示的な中間量として設定する。 数式: `baseline = {` の演算・変換をPythonで評価する。
baseline = {
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、直前の式・構造へ `"mu": 0.3, "horizon_s": 1.0, "mpc_hz": 100,` の要素または終端を対応付ける。
    "mu": 0.3, "horizon_s": 1.0, "mpc_hz": 100,
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、直前の式・構造へ `"wbc_weight_swing": 100.0,` の要素または終端を対応付ける。
    "wbc_weight_swing": 100.0,
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、直前の式・構造へ `"wbc_weight_base": 1.0,` の要素または終端を対応付ける。
    "wbc_weight_base": 1.0,
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、直前の式・構造へ `"wbc_weight_force": 0.01,` の要素または終端を対応付ける。
    "wbc_weight_force": 0.01,
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、直前の式・構造へ `"joint_kp": 0.0, "joint_kd": 3.0,` の要素または終端を対応付ける。
    "joint_kp": 0.0, "joint_kd": 3.0,
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、直前の式・構造へ `}` の要素または終端を対応付ける。
}
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`trial` を後続計算で使う明示的な中間量として設定する。 数式: `trial = baseline | {"wbc_weight_force": 0.03}` の演算・変換をPythonで評価する。
trial = baseline | {"wbc_weight_force": 0.03}
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`changed` を後続計算で使う明示的な中間量として設定する。 数式: `changed = {k: (baseline[k], trial[k]) for k in baseline if baseline[k] !…` の演算・変換をPythonで評価する。
changed = {k: (baseline[k], trial[k]) for k in baseline if baseline[k] != trial[k]}
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`assert len(changed) == 1, "一度に1仮説の規則に反している"` を不変条件として即時検査する。 数式: `assert len(changed) == 1, "一度に1仮説の規則に反している"` の演算・変換をPythonで評価する。
assert len(changed) == 1, "一度に1仮説の規則に反している"
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`changed` をこの章の処理順に沿って実行する。
changed


{'wbc_weight_force': (0.01, 0.03)}

## 数式変更テンプレート

例: 摩擦を等方円錐から異方性ellipseへ変えるなら
\[
\sqrt{(F_x/\mu_x)^2+(F_y/\mu_y)^2}\le F_z
\]
と書き、次を同時に定義する。

- \(\mu_x,\mu_y\) の物理的意味と同定法
- \(F_z<0\) を許さない条件
- smooth化epsilonとgradient
- NMPC側soft constraintとWBC側linear approximationの整合
- flat floorで元式へ戻る回帰test

「式を変える」はC++1行の変更ではなく、model・constraint・solver微分・下位実行・
testの契約変更である。


In [3]:
# 等方円錐と異方性ellipseのmarginを比較する。
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`isotropic_margin` の責務を独立関数として定義する。
def isotropic_margin(fx, fy, fz, mu):
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`return mu*fz - np.hypot(fx, fy)` の値を次の制御境界へ返す。 数式: `return mu*fz - np.hypot(fx, fy)` の演算・変換をPythonで評価する。
    return mu*fz - np.hypot(fx, fy)

# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`anisotropic_margin` の責務を独立関数として定義する。
def anisotropic_margin(fx, fy, fz, mux, muy):
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`return fz - np.sqrt((fx/mux)**2 + (fy/muy)**2)` の値を次の制御境界へ返す。 数式: `return fz - np.sqrt((fx/mux)**2 + (fy/muy)**2)` の演算・変換をPythonで評価する。
    return fz - np.sqrt((fx/mux)**2 + (fy/muy)**2)

# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`test_forces` を後続計算で使う明示的な中間量として設定する。 数式: `test_forces = np.array([[10, 0, 50], [0, 10, 50], [12, 8, 50], [20, 0, 5…` の演算・変換をPythonで評価する。
test_forces = np.array([[10, 0, 50], [0, 10, 50], [12, 8, 50], [20, 0, 50]])
# 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`for f in test_forces:` の反復範囲を固定して各sampleを処理する。
for f in test_forces:
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`old` を後続計算で使う明示的な中間量として設定する。 数式: `old = isotropic_margin(*f, mu=0.3)` の演算・変換をPythonで評価する。
    old = isotropic_margin(*f, mu=0.3)
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`new` を後続計算で使う明示的な中間量として設定する。 数式: `new = anisotropic_margin(*f, mux=0.4, muy=0.2)` の演算・変換をPythonで評価する。
    new = anisotropic_margin(*f, mux=0.4, muy=0.2)
    # 背景: 制御変更はbaselineと残差を固定しないと原因を識別できない。目的: 一変更ずつ比較する再現可能な手順を作るため、`print(f"F={f}: isotropic={old:7.3f}, anisotropic={new:7.3f}")` の観測値を表示して判定根拠を残す。 数式: `print(f"F={f}: isotropic={old:7.3f}, anisotropic={new:7.3f}")` の演算・変換をPythonで評価する。
    print(f"F={f}: isotropic={old:7.3f}, anisotropic={new:7.3f}")


F=[10  0 50]: isotropic=  5.000, anisotropic= 25.000
F=[ 0 10 50]: isotropic=  5.000, anisotropic=  0.000
F=[12  8 50]: isotropic=  0.578, anisotropic=  0.000
F=[20  0 50]: isotropic= -5.000, anisotropic=  0.000


## 最終演習（順番を守る）
1. `task.info`のQ/Rを全24状態・24入力へ対応付ける。
2. 静止4脚でweight-compensating inputと力学残差を計算する。
3. trot 2脚支持でNMPC円錐とWBC pyramid双方のmarginを出す。
4. WBCの各task residualをlogできる設計を作る。
5. policy age watchdogとQP failure fallbackを設計する。
6. Qを1群だけ変え、追従・constraint・torque・solve timeを比較する。
7. 最後に、摩擦または遊脚軌道の式を1つ変更し、元式へ戻る回帰testを書く。

## 修了判定
次を説明できれば、コード変更へ進める。

- x/u/rbd/WBC変数の中身、単位、frame
- Gait、NMPC、WBC、hybrid jointの責務境界
- KFが推定するもの/しないもの
- NMPC円錐とWBC pyramidの差
- weighted WBCでGRF目標がずれる理由
- 100/500 Hz間でpolicyが古くなる危険
- parameter変更と数式変更の検証項目


## 章固有の背景
                複数層のparameterを同時変更すると、改善原因も副作用も同定できない。

                ## 章固有の目的
                症状を残差へ戻し、config変更と式変更を回帰可能な実験として設計する。

                ## この章のASCIIデータフロー
                ```text
                symptom -> identify block/residual -> freeze baseline -> one change
 -> unit/shape/gradient tests -> 30-scenario evidence -> accept or revert
                ```

                ## 上流C++ / faithful pseudocode と数式の行対応
                ```cpp
                // external/legged_control/legged_controllers/config/a1/task.info
Q(state)=...; R(input)=...; mu=0.3; // parameter変更: cost/feasible set
// LeggedInterface::setupOptimalControlProblem
constraint = FrictionConeConstraint(...);     // NMPC式を変更する場所
// WbcBase::formulateFrictionConeTask
D_i * f_i <= 0;                               // WBC近似も同時に整合
// 検査式
old_margin=mu*Fz-hypot(Fx,Fy);                // baseline残差
new_margin=Fz-sqrt((Fx/mux)^2+(Fy/muy)^2);    // 新式と単位を明示
                ```

                **事実のラベル**: `external/legged_control/` の記述はcommit
                `a7f381c0367e98e31c01336e678eef47e304d40d` の上流実装事実。数式展開はそのinterfaceを説明する理論。
                `src/legged_control_mujoco/` に言及した行はproject所有adapterの実装であり、
                ROS1/OCS2 SQP原実装とは同一ではない。

                ## 章固有の結論
                baseline固定、1仮説、物理残差、solver/time、安全指標を揃えて初めて調整になる。式変更はNMPCとWBC双方の整合まで含む。
